# Klassifikation mit allen Werten

`JobSat` als Zielwert


In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectFromModel
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix


## Daten laden

Im One-Hot-Encoded Datensatz sind die One-Hot-Encoded Spalten mit bool Werten. Um es etwas einfacher zu gestalten werden diese Werte hier in Integer-Werte (False -> 0; True -> 1) umgewandelt.

In [13]:

df = pd.read_csv("Abgaben/survey_results_after_clustering.csv")


bool_cols = df.select_dtypes(include=['bool']).columns

for col in bool_cols:
    df[col] = df[col].astype(int)
df.dtypes

ResponseId                          int64
MainBranch                         object
Age                                object
MaxAge                            float64
AgeNum                            float64
EdLevel                            object
Employment                         object
WorkExp                           float64
LearnCodeAI                        object
YearsCode                         float64
DevType                            object
OrgSize                            object
ICorPM                             object
RemoteWork                         object
RemoteCategoryNum                 float64
Industry                           object
AIThreat                           object
NewRole                            object
Country                            object
LanguageChoice                     object
LanguageHaveWorkedWith             object
DatabaseChoice                     object
DatabaseHaveWorkedWith             object
PlatformChoice                    

## Zielvariable in Klassen einteilen 

Dadurch haben wir mehr Trainingsdaten, als wenn wir `JobSat` von 0-10 als Klassen defonieren würden

Klassen
- **Low**: 0–3
- **Medium**: 4–6
- **High**: 7–10


In [16]:
# Nur Zeilen behalten, wo JobSat vorhanden ist
df = df.dropna(subset=["JobSat"]).copy()

def map_jobsat(x):
    x = float(x)
    if x <= 3:
        return "Low"
    elif x <= 6:
        return "Medium"
    else:
        return "High"


## Feature-Spalten bestimmen

- Textspalten: `object` (Strings)
- Numerische Spalten: `int/float`



In [17]:
# Textspalten (Strings)
text_cols = df.select_dtypes(include=["object"]).columns.tolist()

df["__text__"] = df[text_cols].fillna("").agg(" ".join, axis=1)

# Numerische Spalten
num_cols = df.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()

# Zielspalte aus numerischen Features entfernen (falls vorhanden)
num_cols = [c for c in num_cols if c != "JobSat"] #?

df = df.dropna(subset=["__text__"] + num_cols + ["JobSat"])

y = df["JobSat"].apply(map_jobsat)

print("Textspalten:", text_cols)
print("Numerische Spalten:", num_cols)

# Feature-Matrix aus den ausgewählten Spalten
X = df[["__text__"] + num_cols].copy()


X.head()

Textspalten: ['MainBranch', 'Age', 'EdLevel', 'Employment', 'LearnCodeAI', 'DevType', 'OrgSize', 'ICorPM', 'RemoteWork', 'Industry', 'AIThreat', 'NewRole', 'Country', 'LanguageChoice', 'LanguageHaveWorkedWith', 'DatabaseChoice', 'DatabaseHaveWorkedWith', 'PlatformChoice', 'PlatformHaveWorkedWith', 'WebframeChoice', 'WebframeHaveWorkedWith', 'DevEnvsChoice', 'DevEnvsHaveWorkedWith', 'OfficeStackAsyncHaveWorkedWith', 'CommPlatformHaveWorkedWith', 'CommPlatformWantToWorkWith', 'AIModelsChoice', 'AIModelsHaveWorkedWith', 'AISelect', 'AIAgents', 'AIAgent_Uses']
Numerische Spalten: ['ResponseId', 'MaxAge', 'AgeNum', 'WorkExp', 'YearsCode', 'RemoteCategoryNum', 'RemoteMissing', 'ConvertedCompTotal']


,__text__,ResponseId,MaxAge,AgeNum,WorkExp,YearsCode,RemoteCategoryNum,RemoteMissing,ConvertedCompTotal
0,i am a developer by profession 25-34 years old...,1,34.0,29.0,8.0,14.0,0.00,0,61659.84
1,i am a developer by profession 25-34 years old...,2,34.0,29.0,2.0,10.0,0.25,0,105102.00
2,i am a developer by profession 35-44 years old...,4,44.0,39.0,4.0,5.0,0.00,0,36435.36
3,i am a developer by profession 35-44 years old...,5,44.0,39.0,21.0,22.0,0.50,1,60000.00
4,i am a developer by profession 45-54 years old...,6,54.0,49.0,15.0,20.0,0.50,1,120000.00


## Train/Test Split



In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))
print("Train class distribution:\n", y_train.value_counts(normalize=True))
print("Test class distribution:\n", y_test.value_counts(normalize=True))

Train size: 10191
Test size: 2548
Train class distribution:
 JobSat
High      0.720538
Medium    0.218330
Low       0.061132
Name: proportion, dtype: float64
Test class distribution:
 JobSat
High      0.720565
Medium    0.218210
Low       0.061224
Name: proportion, dtype: float64


## Preprocessing für Text und numerische Spalten




In [19]:
preprocessor = ColumnTransformer(
    transformers=[
        ("text", TfidfVectorizer(), "__text__"),
        ("num", StandardScaler(), num_cols)
    ],
    remainder="drop"
)

preprocessor

,transformers,"[('text', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'


## Pipeline definieren

- Preprocessing
- Feature-Selektion (SelectFromModel mit L1-LinearSVC)
- Klassifikator (LinearSVC)


In [20]:
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("feature_selection", SelectFromModel(LinearSVC(penalty="l1", dual=False, C=0.5))),
    ("classifier", LinearSVC())
])

pipeline

,steps,"[('preprocessing', ...), ('feature_selection', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('text', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


## GridSearchCV



In [21]:

parameters = {
    # TF-IDF: nur word, nur die zwei wichtigsten Varianten
    "preprocessing__text__analyzer": ["word"],
    "preprocessing__text__ngram_range": [(1, 1), (1, 2)],
    "preprocessing__text__max_df": [0.9],
    "preprocessing__text__min_df": [2],

    # Klassifikator: 2 sinnvolle Regularisierungen + optional balancing
    "classifier__C": [1.0, 2.0],
    "classifier__class_weight": [None, "balanced"],
}

grid = GridSearchCV(pipeline, param_grid=parameters, verbose=2, cv=3, n_jobs=-1)

grid

,estimator,Pipeline(step...LinearSVC())])
,param_grid,"{'classifier__C': [1.0, 2.0], 'classifier__class_weight': [None, 'balanced'], 'preprocessing__text__analyzer': ['word'], 'preprocessing__text__max_df': [0.9], ...}"
,scoring,None
,n_jobs,-1
,refit,True
,cv,3
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('text', ...), ('num', ...)]"


## Grid Search + Beste Parameter


In [22]:

grid.fit(X_train, y_train)

print("Beste Performance:", grid.best_score_)
print("Beste Parameter:\n", grid.best_params_)

Fitting 3 folds for each of 8 candidates, totalling 24 fits


C:\Users\MoritzSchwarz\PycharmProjects\data-analytics-project\.venv\Lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Beste Performance: 0.7187714650181533
Beste Parameter:
 {'classifier__C': 1.0, 'classifier__class_weight': None, 'preprocessing__text__analyzer': 'word', 'preprocessing__text__max_df': 0.9, 'preprocessing__text__min_df': 2, 'preprocessing__text__ngram_range': (1, 1)}


## Evaluation auf Testdaten


In [24]:
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print("Classification Report (Test):")
print(classification_report(y_test, y_pred))

print("Confusion Matrix (rows=true, cols=pred):")
print(confusion_matrix(y_test, y_pred, labels=["Low", "Medium", "High"]))

Classification Report (Test):
              precision    recall  f1-score   support

        High       0.75      0.97      0.84      1836
         Low       0.50      0.01      0.01       156
      Medium       0.42      0.12      0.19       556

    accuracy                           0.73      2548
   macro avg       0.56      0.37      0.35      2548
weighted avg       0.66      0.73      0.65      2548

Confusion Matrix (rows=true, cols=pred):
[[   1   37  118]
 [   1   69  486]
 [   0   58 1778]]


In [25]:
y_pred_train = best_model.predict(X_train)

print("Classification Report (Train):")
print(classification_report(y_train, y_pred_train))

Classification Report (Train):
              precision    recall  f1-score   support

        High       0.75      0.97      0.85      7343
         Low       0.85      0.02      0.03       623
      Medium       0.47      0.15      0.23      2225

    accuracy                           0.73     10191
   macro avg       0.69      0.38      0.37     10191
weighted avg       0.70      0.73      0.66     10191



In [26]:
# Klassifikation: RemoteCategoryNum als Zielwert

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectFromModel
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix

# -------------------------
# 1) Daten laden
# -------------------------
df = pd.read_csv("Abgaben/survey_results_after_clustering.csv")

# bool -> int (falls vorhanden)
bool_cols = df.select_dtypes(include=["bool"]).columns
df[bool_cols] = df[bool_cols].astype(int)

target_col = "RemoteCategoryNum"

# Target muss vorhanden sein (bei dir ist es i.d.R. komplett gefüllt, aber sicher ist sicher)
df = df.dropna(subset=[target_col]).copy()

# Optional: als Klassen-Labels (string), damit es wirklich als Klassifikation läuft
y = df[target_col].astype(str)

print("Target distribution:\n", y.value_counts())

# -------------------------
# 2) Features bestimmen (WICHTIG: Leakage vermeiden!)
# -------------------------
# KEIN RemoteWork und KEIN RemoteMissing als Feature verwenden
leakage_cols = {"RemoteWork", "RemoteMissing", target_col, "__text__", "cluster"}

# Textspalten (Strings)
text_cols = [c for c in df.select_dtypes(include=["object"]).columns if c not in leakage_cols]

df["__text__"] = df[text_cols].fillna("").agg(" ".join, axis=1)

# Numerische Spalten
num_cols = df.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
num_cols = [c for c in num_cols if c not in leakage_cols]

X = df[["__text__"] + num_cols].copy()

print("Textspalten:", len(text_cols))
print("Numerische Spalten:", len(num_cols))
print("X shape:", X.shape)

# -------------------------
# 3) Train/Test Split
# -------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

# -------------------------
# 4) Preprocessing
# -------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ("text", TfidfVectorizer(), "__text__"),
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), num_cols)
    ],
    remainder="drop"
)

# -------------------------
# 5) Pipeline
# -------------------------
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("feature_selection", SelectFromModel(
        LinearSVC(penalty="l1", dual=False, C=0.5, max_iter=10000)
    )),
    ("classifier", LinearSVC(max_iter=10000))
])

# -------------------------
# 6) GridSearchCV (macro-F1 ist hier sinnvoll)
# -------------------------
parameters = {
    "preprocessing__text__ngram_range": [(1, 1), (1, 2)],
    "preprocessing__text__max_df": [0.9],
    "preprocessing__text__min_df": [2],

    "classifier__C": [0.5, 1.0, 2.0],
    "classifier__class_weight": [None, "balanced"],
}

grid = GridSearchCV(
    pipeline,
    param_grid=parameters,
    verbose=2,
    cv=3,
    n_jobs=-1,
    scoring="f1_macro"  # <- wichtig bei Multi-Class
)

grid.fit(X_train, y_train)

print("Beste Performance (CV, f1_macro):", grid.best_score_)
print("Beste Parameter:\n", grid.best_params_)

# -------------------------
# 7) Evaluation
# -------------------------
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print("\nClassification Report (Test):")
print(classification_report(y_test, y_pred))

labels_sorted = sorted(y.unique(), key=lambda s: float(s))
print("Confusion Matrix (rows=true, cols=pred):")
print(confusion_matrix(y_test, y_pred, labels=labels_sorted))


Target distribution:
 RemoteCategoryNum
0.0     4296
0.5     3197
0.75    2738
0.25    2620
1.0     1836
Name: count, dtype: int64
Textspalten: 30
Numerische Spalten: 7
X shape: (14687, 8)
Train size: 11749
Test size: 2938
Fitting 3 folds for each of 12 candidates, totalling 36 fits
Beste Performance (CV, f1_macro): 0.4381109319099319
Beste Parameter:
 {'classifier__C': 2.0, 'classifier__class_weight': 'balanced', 'preprocessing__text__max_df': 0.9, 'preprocessing__text__min_df': 2, 'preprocessing__text__ngram_range': (1, 1)}

Classification Report (Test):
              precision    recall  f1-score   support

         0.0       0.54      0.68      0.60       859
        0.25       0.34      0.35      0.35       524
         0.5       0.84      0.44      0.58       640
        0.75       0.30      0.26      0.28       548
         1.0       0.36      0.51      0.43       367

    accuracy                           0.47      2938
   macro avg       0.48      0.45      0.45      2938
wei

In [29]:
# Klassifikation – RemoteCategoryNum (gemappt auf Klassen 0–4)
# Ziel: RemoteCategoryNum vorhersagen (5 Klassen) mit Text (TF-IDF) + Numerik (StandardScaler)
# Modell: LinearSVC + Feature Selection (L1)

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectFromModel
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix

## Daten laden

df = pd.read_csv("Abgaben/survey_results_after_clustering.csv")
print("Loaded:", df.shape)

# Bool-Spalten ggf. in int umwandeln (falls vorhanden)
bool_cols = df.select_dtypes(include=["bool"]).columns
for col in bool_cols:
    df[col] = df[col].astype(int)

## Feature-Spalten bestimmen (Text + Numerik)

target_col = "RemoteCategoryNum"

# 1) Target muss vorhanden sein
df = df.dropna(subset=[target_col]).copy()

# -------------------------------------------------------------------
# ✅ CHANGE 1: Leakage-Spalten definieren und aus Text/NumFeatures werfen
# -------------------------------------------------------------------
LEAK_COLS = {
    "RemoteWork",          # direkte Quelle für RemoteCategoryNum
    target_col,            # zur Sicherheit (falls als object vorhanden)
    "cluster",             # falls vorhanden (aus Clustering)
}
# Optional: wenn du derived columns hast, hier ergänzen:
# LEAK_COLS |= {"RemoteMissing"}  # nur falls das zu eng am Target hängt (bei dir eher ok)

# Textspalten (Strings) – Leakage rausnehmen
text_cols = df.select_dtypes(include=["object"]).columns.tolist()
text_cols = [c for c in text_cols if c not in LEAK_COLS]  # ✅ CHANGE

# Text pro Zeile zu einem Dokument zusammenbauen
df["__text__"] = df[text_cols].fillna("").agg(" ".join, axis=1)

# -------------------------------------------------------------------
# ✅ CHANGE 2 (optional): Quick Leakage Check im Text
# -------------------------------------------------------------------
# Wenn diese Tokens massenhaft vorkommen, hast du sehr wahrscheinlich noch Leakage drin.
leak_tokens = ["remote", "hybrid", "in-person", "in person", "your choice"]
hits = {t: df["__text__"].str.lower().str.contains(t).mean() for t in leak_tokens}
print("Leak-token share in __text__ (sollte NICHT extrem hoch sein):\n", hits)

# Numerische Spalten
num_cols = df.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()

# Target & Leakage aus Numerik entfernen
num_cols = [c for c in num_cols if c not in LEAK_COLS]  # ✅ CHANGE

# Optional: ID-Spalten entfernen, falls vorhanden
for maybe_id in ["ResponseId"]:
    if maybe_id in num_cols:
        num_cols.remove(maybe_id)

print("Textspalten:", len(text_cols))
print("Numerische Spalten:", len(num_cols))

# 2) NaNs für Features behandeln
# (bei Numerik droppen ist ok, aber kostet Daten; alternativ: Imputer in Pipeline)
df = df.dropna(subset=num_cols).copy()

## Target vorbereiten: RemoteCategoryNum -> Klassen 0–4 (NACH den Drops!)

df[target_col] = df[target_col].astype(float).round(2)

# -------------------------------------------------------------------
# ✅ CHANGE 3: sauberes Mapping auf 0..4 (robust gegen kleine Rundungsfehler)
# -------------------------------------------------------------------
# Erwartet: RemoteCategoryNum in {0, 0.25, 0.5, 0.75, 1.0}
# Wir mappen: 0->0, 0.25->1, 0.5->2, 0.75->3, 1.0->4
y = (df[target_col] * 4).round().clip(0, 4).astype(int).astype(str)  # ✅ CHANGE

print("Target distribution:\n", y.value_counts().sort_index())

## X bauen

X = df[["__text__"] + num_cols].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

# Safety-Check (optional)
assert len(X) == len(y), f"Mismatch: X={len(X)} vs y={len(y)}"

## Train/Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))
print("Train class distribution:\n", y_train.value_counts(normalize=True).sort_index())
print("Test class distribution:\n", y_test.value_counts(normalize=True).sort_index())

## Preprocessing (TF-IDF für Text, StandardScaler für Numerik)

preprocessor = ColumnTransformer(
    transformers=[
        ("text", TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 1),
            min_df=2,
            max_df=0.9
        ), "__text__"),
        ("num", StandardScaler(), num_cols),
    ],
    remainder="drop"
)

## Pipeline

pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("feature_selection", SelectFromModel(
        LinearSVC(penalty="l1", dual=False, C=0.5, max_iter=20000)
    )),
    ("classifier", LinearSVC(max_iter=20000))
])

## GridSearchCV (auf f1_macro optimieren)

parameters = {
    "preprocessing__text__ngram_range": [(1, 1), (1, 2)],
    "preprocessing__text__min_df": [2, 5],
    "preprocessing__text__max_df": [0.9],

    "classifier__C": [0.5, 1.0, 2.0],
    "classifier__class_weight": [None, "balanced"],
}

grid = GridSearchCV(
    pipeline,
    param_grid=parameters,
    scoring="f1_macro",
    verbose=2,
    cv=3,
    n_jobs=-1
)

## Training

grid.fit(X_train, y_train)

print("Beste Performance (CV, f1_macro):", grid.best_score_)
print("Beste Parameter:\n", grid.best_params_)

## Evaluation

best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

labels_sorted = ["0", "1", "2", "3", "4"]

print("\nClassification Report (Test):")
print(classification_report(y_test, y_pred, labels=labels_sorted))

print("Confusion Matrix (rows=true, cols=pred):")
print(confusion_matrix(y_test, y_pred, labels=labels_sorted))

## Optional: Train-Report (Overfitting-Check)

y_pred_train = best_model.predict(X_train)

print("\nClassification Report (Train):")
print(classification_report(y_train, y_pred_train, labels=labels_sorted))


Loaded: (14687, 40)
Leak-token share in __text__ (sollte NICHT extrem hoch sein):
 {'remote': np.float64(0.0), 'hybrid': np.float64(0.0), 'in-person': np.float64(0.0), 'in person': np.float64(0.0), 'your choice': np.float64(0.0)}
Textspalten: 30
Numerische Spalten: 7
Target distribution:
 RemoteCategoryNum
0    3925
1    2283
2    2761
3    2317
4    1453
Name: count, dtype: int64
X shape: (12739, 8)
y shape: (12739,)
Train size: 10191
Test size: 2548
Train class distribution:
 RemoteCategoryNum
0    0.308115
1    0.179178
2    0.216760
3    0.181925
4    0.114022
Name: proportion, dtype: float64
Test class distribution:
 RemoteCategoryNum
0    0.308085
1    0.179356
2    0.216641
3    0.181711
4    0.114207
Name: proportion, dtype: float64
Fitting 3 folds for each of 24 candidates, totalling 72 fits
Beste Performance (CV, f1_macro): 0.4360787718172032
Beste Parameter:
 {'classifier__C': 0.5, 'classifier__class_weight': 'balanced', 'preprocessing__text__max_df': 0.9, 'preprocessing__te